## Problem Statement

### Business Context

The healthcare industry is rapidly evolving, with professionals facing increasing challenges in managing vast volumes of medical data while delivering accurate and timely diagnoses. The need for quick access to comprehensive, reliable, and up-to-date medical knowledge is critical for improving patient outcomes and ensuring informed decision-making in a fast-paced environment.

Healthcare professionals often encounter information overload, struggling to sift through extensive research and data to create accurate diagnoses and treatment plans. This challenge is amplified by the need for efficiency, particularly in emergencies, where time-sensitive decisions are vital. Furthermore, access to trusted, current medical information from renowned manuals and research papers is essential for maintaining high standards of care.

To address these challenges, healthcare centers can focus on integrating systems that streamline access to medical knowledge, provide tools to support quick decision-making, and enhance efficiency. Leveraging centralized knowledge platforms and ensuring healthcare providers have continuous access to reliable resources can significantly improve patient care and operational effectiveness.

**Common Questions to Answer**

**1. Diagnostic Assistance**: "What are the common symptoms and treatments for pulmonary embolism?"

**2. Drug Information**: "Can you provide the trade names of medications used for treating hypertension?"

**3. Treatment Plans**: "What are the first-line options and alternatives for managing rheumatoid arthritis?"

**4. Specialty Knowledge**: "What are the diagnostic steps for suspected endocrine disorders?"

**5. Critical Care Protocols**: "What is the protocol for managing sepsis in a critical care unit?"

### Objective

As an AI specialist, your task is to develop a RAG-based AI solution using renowned medical manuals to address healthcare challenges. The objective is to **understand** issues like information overload, **apply** AI techniques to streamline decision-making, **analyze** its impact on diagnostics and patient outcomes, **evaluate** its potential to standardize care practices, and **create** a functional prototype demonstrating its feasibility and effectiveness.

### Data Description

The **Merck Manuals** are medical references published by the American pharmaceutical company Merck & Co., that cover a wide range of medical topics, including disorders, tests, diagnoses, and drugs. The manuals have been published since 1899, when Merck & Co. was still a subsidiary of the German company Merck.

The manual is provided as a PDF with over 4,000 pages divided into 23 sections.

## Installing and Importing Necessary Libraries and Dependencies

In [ ]:
# Installation for GPU llama-cpp-python
# uncomment and run the following code in case GPU is being used
!CMAKE_ARGS="-DLLAMA_CUBLAS=on" FORCE_CMAKE=1 pip install llama-cpp-python==0.2.28 --force-reinstall --no-cache-dir -q

# Installation for CPU llama-cpp-python
# uncomment and run the following code in case GPU is not being used
# !CMAKE_ARGS="-DLLAMA_CUBLAS=off" FORCE_CMAKE=1 pip install llama-cpp-python==0.2.28 --force-reinstall --no-cache-dir -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 212.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 262.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 256.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 302.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry-exporter-otlp-proto-common==1.38.0, but you have opentelemetry-exporter-otlp-proto-common 1.42.1 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry-proto==1.38.0, but you have opentelemetry-proto 1.42.1 which is incom

**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [ ]:
!pip install huggingface_hub==0.35.3 pandas==2.2.2 tiktoken==0.12.0 pymupdf==1.26.5 langchain==0.3.27 langchain-community==0.3.31 chromadb==1.1.1 sentence-transformers==5.1.1 numpy==2.3.3 -q


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.3.3 which is incompatible.


**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [ ]:
#Libraries for processing dataframes,text
import json,os
import tiktoken
import pandas as pd

#Libraries for Loading Data, Chunking, Embedding, and Vector Databases
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from langchain_community.vectorstores import Chroma

#Libraries for downloading and loading the llm
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

## Question Answering using LLM

#### Downloading and Loading the model

In [ ]:
model_name_or_path = "TheBloke/Mistral-7B-Instruct-v0.2-GGUF"
model_basename = "mistral-7b-instruct-v0.2.Q6_K.gguf"

In [ ]:
model_path = hf_hub_download(
    repo_id= model_name_or_path,
    filename= model_basename
)

mistral-7b-instruct-v0.2.Q6_K.gguf:   0%|          | 0.00/5.94G [00:00<?, ?B/s]

In [ ]:
#uncomment the below snippet of code if the runtime is connected to GPU.
llm = Llama(
    model_path=model_path,
    n_ctx=2300,
    n_gpu_layers=36,
    n_batch=512
)

AVX = 1 | AVX_VNNI = 0 | AVX2 = 1 | AVX512 = 1 | AVX512_VBMI = 0 | AVX512_VNNI = 0 | FMA = 1 | NEON = 0 | ARM_FMA = 0 | F16C = 1 | FP16_VA = 0 | WASM_SIMD = 0 | BLAS = 1 | SSE3 = 1 | SSSE3 = 1 | VSX = 0 | 


#### Response

In [ ]:
def response(query,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    model_output = llm(
      prompt=query,
      max_tokens=max_tokens,
      temperature=temperature,
      top_p=top_p,
      top_k=top_k
    )

    return model_output['choices'][0]['text']

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
user_input_1 = "What is the protocol for managing sepsis in a critical care unit?"
response(user_input_1)

'\n\nSepsis is a life-threatening condition that can arise from an infection, and it requires prompt recognition and aggressive management in a critical care unit. The following are general steps for managing sepsis in a critical care unit:\n\n1. Early recognition: Recognize the signs and symptoms of sepsis early and initiate treatment as soon as possible. Sepsis can present with various clinical features, including fever or hypothermia, tachycardia or bradycardia, altered mental status, respiratory distress, and lactic acidosis.\n2. ABCs'

* Pass/Fail Grade: FAIL

* The answer provides context on sepsis but does not describe the protocol for managing sepsis. Early recognition, while useful, is not helpful once a sepsis patient has arrived in the critical care unit.



### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
user_input_2 = "What are the common symptoms for appendicities, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
response(user_input_2)

Llama.generate: prefix-match hit


'\n\nThe symptoms of appendicitis include:\n\n1. Abdominal pain, usually starting around the navel area but eventually moving to the lower right side of the abdomen. The pain may be constant or come and go.\n2. Loss of appetite and feeling sick to your stomach (nausea).\n3. Vomiting.\n4. Fever, which may start after the other symptoms appear.\n5. Diarrhea or constipation.\n6. Abdominal swelling and rigidity.\n7. Inability to pass gas or have a bowel movement.'

* Pass/Fail Grade: FAIL

* The answer correctly handles the first part of the question, detailing common symptoms of appendicitis. The second question is not addressed in any capacity.

* If we increase max_tokens, it is possible we may get an answer to the second part of the question.

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
user_input_3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
response(user_input_3)

Llama.generate: prefix-match hit


"\n\nSudden patchy hair loss, also known as alopecia areata, is a common autoimmune disorder that affects the hair follicles. It can result in round or oval bald patches on the scalp, but it can also occur on other parts of the body such as the beard area, eyebrows, and eyelashes.\n\nThe exact cause of alopecia areata is not known, but it's believed to be related to a problem with the immune system. Some possible triggers for this condition include stress, genetics, viral infections, and certain medications."

* Pass/Fail Grade: FAIL

* The answer provides alternative names for patchy hair loss before detailing potential symptoms, failing to address treatments or solutions. The second half of the answer correctly articules possible causes behind patchy hair loss.

* Current top_p settings enable the model to consider a wide pool of words when predicting the next token. It's worth exploring if we are giving the model too much freedom.

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
user_input_4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
response(user_input_4)

Llama.generate: prefix-match hit


'\n\nA person who has sustained a physical injury to the brain tissue may require various treatments depending on the severity and location of the injury. Here are some common treatments that may be recommended:\n\n1. Emergency care: In case of a traumatic brain injury (TBI), it is essential to seek emergency medical attention as soon as possible. The primary goal of emergency care is to prevent further damage to the brain, stabilize vital signs, and manage any life-threatening conditions.\n2. Medications: Depending on the symptoms, healthcare professionals may prescribe medications to manage various conditions associated with a'

* Pass/Fail Grade: PASS

* The answer describes potential treatments for a person who has sustained a physical injury to brain tissue.

* Increasing max_tokens may enable the model to provide a longer list of recommendations. Given current constraints, dictating the format or limiting scope may help the model become more succint when answering.

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
user_input_5 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
response(user_input_5)

Llama.generate: prefix-match hit


"\n\nFirst and foremost, if you suspect that someone has fractured their leg while hiking, it's essential to ensure their safety and prevent further injury. Here are some necessary precautions:\n\n1. Keep the person calm and still: Encourage them to remain as still as possible to minimize pain and prevent worsening the injury.\n2. Assess the situation: Check for any signs of shock, such as pale skin, rapid heartbeat, or shallow breathing. If you notice these symptoms, seek medical help immediately.\n3. Immobilize the leg: Use a splint, sl"

* Pass/Fail Grade: PASS

* The answer successfully covers necessary precautions and treatments for a person who fractured their leg.



## Question Answering using LLM with Prompt Engineering

In [ ]:
system_prompt = "You are an AI assistant specializing in medical triage. Your role is to assess patient symptoms, provide initial guidance, and help determine the urgency of their condition. Focus on accuracy, clarity, safety, and brevity. If a user's prompt involves multiple questions, answer the questions in order. Do not provide definitive diagnoses or treatment plans, but rather guide users on next steps such as seeking professional medical attention. Always prioritize patient well-being."

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
user_input_1 = system_prompt+"\n"+ "What is the protocol for managing sepsis in a critical care unit?"
response(user_input_1, max_tokens=256)

Llama.generate: prefix-match hit


'\nSepsis is a life-threatening condition that requires immediate medical attention. In a critical care unit, managing sepsis involves several key steps:\n1. Recognition and early identification: Look for signs of infection, such as fever, chills, rapid heart rate, rapid breathing, confusion, or low blood pressure. Use the Sequential Organ Failure Assessment (SOFA) score to evaluate organ dysfunction.\n2. Immediate resuscitation: Start intravenous fluids and oxygen therapy to maintain adequate blood pressure and oxygenation. Administer broad-spectrum antibiotics as soon as possible based on suspected infection site and culture results.\n3. Source control: Identify and address the source of infection, such as removing an infected catheter or draining an abscess.\n4. Hemodynamic support: Use vasopressors to maintain blood pressure if needed, and consider using inotropes to improve cardiac output.\n5. Renal replacement therapy: Consider initiating renal replacement therapy (dialysis) for 

* Pass/Fail Grade: PASS

* After adding a prompt and increasing max_tokens from 128 to 256, the model provides an answer that begins detailing the protocol for managing sepsis in a critical care unit.

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
user_input_2 = system_prompt+"\n"+ "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
response(user_input_2, temperature=0.1, max_tokens=256)

Llama.generate: prefix-match hit


'\n\nAppendicitis is a medical condition characterized by inflammation of the appendix, a small pouch that extends from the large intestine. Common symptoms include:\n1. Sudden pain in the lower right abdomen, which may start as mild and then gradually worsen over hours.\n2. Loss of appetite and feeling sick to your stomach (nausea).\n3. Vomiting.\n4. Fever, often above 101 degrees Fahrenheit (38.3 degrees Celsius).\n5. Abdominal swelling and rigidity.\n6. Constipation or diarrhea.\n7. Pain that worsens when you walk, cough, or make other jarring movements.\n\nAppendicitis cannot be cured via medicine alone. If left untreated, the appendix may burst, leading to a potentially life-threatening infection in the abdominal cavity called peritonitis. The standard treatment for appendicitis is surgical removal of the appendix, known as an appendectomy. This procedure can be performed laparoscopically (using small incisions and a camera) or open (larger incision).'

* Pass/Fail Grade: PASS

* Adding the temperature paramater brings us the best answer so far; the model answers both questions correctly and succintly.

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
user_input_3 = system_prompt+"\n"+ "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
response(user_input_3, temperature=0.1, max_tokens=256, top_p=0.8)

Llama.generate: prefix-match hit


"\nI'm an AI and don't have the ability to diagnose or provide definitive treatments. However, I can suggest some common causes of sudden patchy hair loss, also known as alopecia areata, and potential solutions that might help.\n\nPossible causes:\n1. Alopecia Areata: An autoimmune disease that causes hair loss in small patches on the scalp, face, or other parts of the body. It often results in round bald spots.\n2. Stress: Sudden stress can trigger alopecia areata or worsen existing conditions.\n3. Vitamin deficiencies: Lack of essential vitamins like iron, zinc, biotin, and vitamin D can lead to hair loss.\n4. Hormonal imbalances: Thyroid disorders, polycystic ovary syndrome (PCOS), or menopause can cause sudden hair loss.\n5. Medications: Certain medications like chemotherapy drugs, antidepressants, and blood thinners can lead to hair loss.\n6. Trauma or injury: Physical trauma or injury to the scalp can result in patchy hair loss.\n\n"

* Pass/Fail Grade: PASS

* The model correctly follows the system prompt, declining to provide treatment advice in the first part of the question. When to user asks for possible causes, the model provides a list of potential reasons.

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
user_input_4 = system_prompt+"\n"+ "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
response(user_input_4, max_tokens=256, top_p=0.5)

Llama.generate: prefix-match hit


"\nI'm an AI and don't have the ability to provide definitive diagnoses or recommend specific treatments. However, I can suggest some general approaches that may be helpful for someone with a brain injury. \n1. Medical Attention: Seek immediate medical attention if you suspect a brain injury. Brain injuries can range from mild to severe and require proper evaluation by healthcare professionals.\n2. Rest and Recovery: Allow the brain time to heal. This may involve staying in bed or limiting physical activity for an extended period.\n3. Rehabilitation: Depending on the severity of the injury, rehabilitation may be necessary to help restore lost functions. This could include occupational therapy, speech therapy, physical therapy, and cognitive rehabilitation.\n4. Medications: Various medications may be prescribed to manage symptoms associated with a brain injury, such as pain, swelling, or seizures.\n5. Supportive Care: Provide emotional support and assistance with daily activities during

* Pass/Fail Grade: PASS

* The model again correctly declines to provide treatment advice and instead suggests helpful approaches related to brain injuries. Reducing top_p from 0.8 to 0.5 does not appear to impact answer quality.

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
user_input_5 = system_prompt+"\n"+ "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
response(user_input_5, temperature=0.5, max_tokens=256, top_p=0.9)

Llama.generate: prefix-match hit


'\n1. Assess the severity of the fracture: Encourage the person to remain calm and still to minimize discomfort and potential damage. If they are experiencing severe pain, swelling, or an inability to move the leg, it may be a sign of a more serious fracture that requires immediate medical attention.\n2. Immobilize the affected leg: Use a splint, sling, or other immobilizing device to help prevent further injury and provide support. This can help reduce pain and swelling, as well as promote proper healing.\n3. Control bleeding: If there is visible bleeding, apply direct pressure with a clean cloth to help stop the flow of blood. Elevate the leg if possible to reduce swelling and promote better circulation.\n4. Seek professional medical attention: Encourage the person to seek medical care as soon as possible, especially if they are experiencing severe pain, swelling, or an inability to move the leg. Depending on the severity of the fracture, they may need to be transported to a hospital

* Pass/Fail Grade: FAIL

* By setting high thresholds for temperature and top_p, the model ignores the system prompt and provides a direct treatment plan for a person who has fractured their leg during a hiking trip.

* The model gives a detailed answer on treatment steps but does not mention care and recovery steps. If max_tokens increased, it's possible the model may detail plans for this step.

## Data Preparation for RAG

In [ ]:
#Libraries for processing dataframes,text
import json,os
import tiktoken
import pandas as pd

#Libraries for Loading Data, Chunking, Embedding, and Vector Databases
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from langchain_community.vectorstores import Chroma

### Loading the Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
manual_pdf_path = "medical_diagnosis_manual.pdf"

In [ ]:
pdf_loader = PyMuPDFLoader(manual_pdf_path)

In [ ]:
manual = pdf_loader.load()

### Data Overview

#### Checking the first 5 pages

In [ ]:
for i in range(5):
    print(f"Page Number : {i+1}",end="\n")
    print(manual[i].page_content,end="\n")

Page Number : 1
travispollitt.31@gmail.com
TSVCXFQGPB
This file is meant for personal use by travispollitt.31@gmail.com only.
Sharing or publishing the contents in part or full is liable for legal action.
Page Number : 2
travispollitt.31@gmail.com
TSVCXFQGPB
This file is meant for personal use by travispollitt.31@gmail.com only.
Sharing or publishing the contents in part or full is liable for legal action.
Page Number : 3
Table of Contents
1
Front    ................................................................................................................................................................................................................
1
Cover    .......................................................................................................................................................................................................
2
Front Matter    ...........................................................................................................

#### Checking the number of pages

In [ ]:
len(manual)

4114

### Data Chunking

In [ ]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='cl100k_base',
    chunk_size=500,
    chunk_overlap=50
)

In [ ]:
document_chunks = pdf_loader.load_and_split(text_splitter)

In [ ]:
len(document_chunks)

8906

In [ ]:
document_chunks[0].page_content

'travispollitt.31@gmail.com\nTSVCXFQGPB\nThis file is meant for personal use by travispollitt.31@gmail.com only.\nSharing or publishing the contents in part or full is liable for legal action.'

In [ ]:
document_chunks[1].page_content

'travispollitt.31@gmail.com\nTSVCXFQGPB\nThis file is meant for personal use by travispollitt.31@gmail.com only.\nSharing or publishing the contents in part or full is liable for legal action.'

In [ ]:
document_chunks[2].page_content

'Table of Contents\n1\nFront    ................................................................................................................................................................................................................\n1\nCover    .......................................................................................................................................................................................................\n2\nFront Matter    ...........................................................................................................................................................................................\n53\n1 - Nutritional Disorders    ...............................................................................................................................................................\n53\nChapter 1. Nutrition: General Considerations    ...........................................................................................

In [ ]:
document_chunks[3].page_content

"275\nChapter 23. Approach to the Patient With Liver Disease    ...........................................................................................\n294\nChapter 24. Testing for Hepatic & Biliary Disorders    ......................................................................................................\n305\nChapter 25. Drugs & the Liver    ................................................................................................................................................\n308\nChapter 26. Alcoholic Liver Disease    ....................................................................................................................................\n314\nChapter 27. Fibrosis & Cirrhosis    ............................................................................................................................................\n322\nChapter 28. Hepatitis    ........................................................................................................

### Embedding

In [ ]:
embedding_model = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")

/tmp/ipykernel_7586/3002436811.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
embedding_1 = embedding_model.embed_query(document_chunks[0].page_content)
embedding_2 = embedding_model.embed_query(document_chunks[1].page_content)

In [ ]:
print("Dimension of the embedding vector ",len(embedding_1))
len(embedding_1)==len(embedding_2)

Dimension of the embedding vector  384


True

In [ ]:
embedding_1,embedding_2

([-0.06196361035108566,
  -0.0031030394602566957,
  -0.017372064292430878,
  -0.030083930119872093,
  0.06954284757375717,
  -0.006777827627956867,
  0.05976992845535278,
  0.012969362549483776,
  -0.023185385391116142,
  0.06111597269773483,
  0.061539944261312485,
  0.06033532693982124,
  0.058779604732990265,
  -0.04049883410334587,
  -0.01056365855038166,
  -0.01210861001163721,
  -0.09679734706878662,
  -0.01599893718957901,
  -0.061375346034765244,
  -0.009709914214909077,
  -0.06623858213424683,
  0.01847759075462818,
  0.013660546392202377,
  0.004169490188360214,
  0.014406890608370304,
  0.015043014660477638,
  -0.06806603074073792,
  0.022776640951633453,
  -0.02842208556830883,
  -0.053441260010004044,
  -0.014721198007464409,
  0.041485272347927094,
  0.030856365337967873,
  0.006668320391327143,
  0.040842242538928986,
  -0.00928849633783102,
  -0.020216893404722214,
  -0.008219187147915363,
  -0.07600369304418564,
  -0.04997728019952774,
  -0.0001889770064735785,
  -0.05

### Vector Database

In [ ]:
out_dir = 'medical_db'

if not os.path.exists(out_dir):
  os.makedirs(out_dir)

In [ ]:
vectorstore = Chroma.from_documents(
    document_chunks,
    embedding_model,
    persist_directory=out_dir
)

In [ ]:
vectorstore = Chroma(persist_directory=out_dir,embedding_function=embedding_model)

/tmp/ipykernel_7586/2756559696.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectorstore = Chroma(persist_directory=out_dir,embedding_function=embedding_model)


In [ ]:
vectorstore.embeddings

HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
), model_name='all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

In [ ]:
vectorstore.similarity_search("What are the common symptoms for appendicitis?",k=3)

[Document(metadata={'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'trapped': '', 'creationDate': 'D:20120615054440Z', 'creator': 'Atop CHM to PDF Converter', 'creationdate': '2012-06-15T05:44:40+00:00', 'file_path': 'medical_diagnosis_manual.pdf', 'modDate': 'D:20260520015030Z', 'format': 'PDF 1.7', 'total_pages': 4114, 'page': 173, 'title': 'The Merck Manual of Diagnosis & Therapy, 19th Edition', 'author': '', 'subject': '', 'source': 'medical_diagnosis_manual.pdf', 'moddate': '2026-05-20T01:50:30+00:00', 'keywords': ''}, page_content="Etiology\nAppendicitis is thought to result from obstruction of the appendiceal lumen, typically by lymphoid\nhyperplasia, but occasionally by a fecalith, foreign body, or even worms. The obstruction leads to\ndistention, bacterial overgrowth, ischemia, and inflammation. If untreated, necrosis, gangrene, and\nperforation occur. If the perforation is contained by the omentum, an appendiceal abscess results.\nSymptoms and Signs\nThe classic 

### Retriever

In [ ]:
retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 3}
)

In [ ]:
rel_docs = retriever.get_relevant_documents("What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?")
rel_docs

/tmp/ipykernel_7586/1878035297.py:1: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  rel_docs = retriever.get_relevant_documents("What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?")


[Document(metadata={'source': 'medical_diagnosis_manual.pdf', 'subject': '', 'creationdate': '2012-06-15T05:44:40+00:00', 'creator': 'Atop CHM to PDF Converter', 'format': 'PDF 1.7', 'moddate': '2026-05-20T01:50:30+00:00', 'modDate': 'D:20260520015030Z', 'file_path': 'medical_diagnosis_manual.pdf', 'trapped': '', 'author': '', 'page': 3390, 'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'total_pages': 4114, 'title': 'The Merck Manual of Diagnosis & Therapy, 19th Edition', 'creationDate': 'D:20120615054440Z', 'keywords': ''}, page_content='(eg, meniscal tears, cartilaginous injuries). Arteriography may be necessary for suspected arterial injuries\n(eg, some popliteal artery injuries). Nerve conduction studies may be indicated for nerve injuries.\nTreatment\n• Treatment of life- or limb-threatening injuries\n• Splinting\n• Definitive treatment (eg, reduction) for certain injuries\n• Rest, ice, compression, and elevation (RICE)\n• Usually immobilization\nIn the emergency depa

In [ ]:
model_output = llm(
      "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?", #Complete the code to pass the query
      max_tokens=256,
      temperature=0.1,
    )

In [ ]:
model_output['choices'][0]['text']

"\n\nFirst and foremost, if you suspect that someone has fractured their leg while hiking, it is essential to ensure their safety and prevent further injury. Here are some necessary precautions:\n\n1. Keep the person calm and still: Encourage them to remain as still as possible to minimize pain and prevent worsening the injury.\n2. Assess the situation: Check for any signs of shock, such as pale skin, rapid heartbeat, or shallow breathing. If you notice these symptoms, seek medical help immediately.\n3. Immobilize the leg: Use a splint, sling, or other available materials to immobilize the leg to prevent movement and further injury. Be sure not to apply too much pressure on the injured area, as this can cause additional pain or damage.\n4. Provide warmth: Keep the injured leg warm to promote blood flow and reduce discomfort.\n5. Seek medical help: If possible, contact emergency services or arrange for transportation to a hospital for further evaluation and treatment.\n\nOnce you have e

### System and User Prompt Template

In [ ]:
qna_system_message = """
You are an AI assistant whose sole purpose is to answer user questions based *strictly and exclusively* on the provided `###Context`, which contains information from the Merck Medical Manuals. Do not use any external knowledge.

Since you are providing information related to medical topics, it is imperative that you *never* offer medical advice, diagnose conditions, or suggest treatments. Always include a disclaimer at the beginning of your response stating that users should consult a qualified healthcare professional for medical advice regarding their specific condition. Focus on accuracy, clarity, safety, and brevity.

If the user's `###Question` contains multiple parts, answer each part comprehensively and in sequence, adhering to the provided context.

If the `###Context` does not contain sufficient information to answer the `###Question`, state: "The information to answer this question is not available in the provided context." Do not generate speculative answers.

Do not explicitly refer to the 'context' or 'documents' in your final answer body.
"""

In [ ]:
qna_user_message_template = """
Please use the following context to answer the user's question. Do not use any other information. If the answer is not in the context, state that the information is not available.

Context:
{context}

Question:
{question}
"""

### Response Function

In [ ]:
def generate_rag_response(user_input,k=3,max_tokens=512,temperature=0,top_p=0.95,top_k=50):
    global qna_system_message,qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks = retriever.get_relevant_documents(query=user_input,k=k)
    context_list = [d.page_content for d in relevant_document_chunks]

    # Combine document chunks into a single context
    context_for_query = ". ".join(context_list)

    user_message = qna_user_message_template.replace('{context}', context_for_query)
    user_message = user_message.replace('{question}', user_input)

    prompt = qna_system_message + '\n' + user_message

    # Generate the response
    try:
        response = llm(
                  prompt=prompt,
                  max_tokens=max_tokens,
                  temperature=temperature,
                  top_p=top_p,
                  top_k=top_k
                  )

        # Extract and print the model's response
        response = response['choices'][0]['text'].strip()
    except Exception as e:
        response = f'Sorry, I encountered the following error: \n {e}'

    return response

## Question Answering using RAG

### Query 1: What is the protocol for managing sepsis in a critical care unit?


In [ ]:
user_input_1 = "What is the protocol for managing sepsis in a critical care unit?"
generate_rag_response(user_input_1)

Llama.generate: prefix-match hit


"Answer:\nIn the context provided, the management of sepsis in a critical care unit involves several steps. If sepsis or septic shock is suspected, cultures should be obtained of blood and any other appropriate specimens. Empiric antibiotics are given after these cultures are obtained, with early treatment appearing to improve survival. Antibiotics should be adjusted according to the results of culture and susceptibility testing, surgical drainage of abscesses may be necessary, and internal devices that are suspected sources of bacteria should usually be removed. Continuous monitoring of physiologic parameters is essential, with ongoing alarms strictly followed. Routine daily blood tests, including electrolytes and a CBC, as well as tests for arrhythmias and liver enzymes in patients receiving TPN, are recommended to help detect problems early. Point-of-care testing can be used to do certain blood tests at the patient's bedside or unit. Patients with sepsis or septic shock typically ha

* Pass/Fail Grade: PASS

* The model provides a clear answer that appears to be from the medical diagnosis manual.

* Despite a higher max_tokens limit, the model fails to concisely wrap up its answer. Further experimentation needed.

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
user_input_2 = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
generate_rag_response(user_input_2)

Llama.generate: prefix-match hit


"Answer:\nThe common symptoms of appendicitis include epigastric or periumbilical pain followed by brief nausea, vomiting, and anorexia. After a few hours, the pain shifts to the right lower quadrant. Pain increases with cough and motion. Classic signs are right lower quadrant direct and rebound tenderness located at McBurney's point, as well as pain felt in the right lower quadrant with palpation of the left lower quadrant (Rovsing sign), an increase in pain from passive extension of the right hip joint that stretches the iliopsoas muscle (psoas sign), or pain caused by passive internal rotation of the flexed thigh (obturator sign). Low-grade fever is common. However, these classic findings appear in less than 50% of patients, and many variations of symptoms and signs occur.\n\nAppendicitis cannot be cured via medicine alone. The standard treatment for appendicitis is surgical removal of the appendix through an open or laparoscopic appendectomy. This is necessary to prevent complicati

* Pass/Fail Grade: PASS

* The model answers both questions correctly with information that appears to be from the manual. It remains unable to complete its answer with available tokens.

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
user_input_3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
generate_rag_response(user_input_3)

Llama.generate: prefix-match hit


"Answer:\nSudden patchy hair loss, also known as alopecia areata, is a common autoimmune disorder that affects genetically susceptible individuals. The exact cause of this condition is unclear but it's believed to involve environmental triggers and inflammation targeting the hair follicles.\n\nThe treatment options for alopecia areata include:\n1. Topical corticosteroids: These medications can help reduce inflammation and stimulate hair growth. They are applied directly to the affected area.\n2. Intralesional corticosteroid injections: Corticosteroids can also be injected into the affected area for more effective treatment.\n3. Systemic corticosteroids: In severe cases, oral corticosteroids may be prescribed to help reduce inflammation and stimulate hair growth.\n4. Topical minoxidil: This medication can help promote hair growth by increasing blood flow to the affected area.\n5. Topical immunotherapy: Diphencyprone or squaric acid dibutylester can be used to stimulate an immune respons

* Pass/Fail Grade: FAIL

* While the model answers both questions correctly, it answers them in the wrong order and possibly violates its system prompt. Deeper explanation is needed to determine if the model a) wrongly provided treatment advice or b) simply read treatment advice from the manual.

### Fine-tuning

### Query 1: What is the protocol for managing sepsis in a critical care unit?


In [ ]:
user_input_1 = "What is the protocol for managing sepsis in a critical care unit?"
generate_rag_response(user_input_1,max_tokens=1024)

Llama.generate: prefix-match hit


"Answer:\nIn the context provided, the management of sepsis in a critical care unit involves several steps. If sepsis or septic shock is suspected, cultures are obtained of blood and any other appropriate specimens. Empiric antibiotics are given after appropriate cultures are obtained, with early treatment appearing to improve survival. Antibiotics should be adjusted according to the results of culture and susceptibility testing, surgical drainage of abscesses may be necessary, and internal devices suspected as sources of bacteria should usually be removed. It is important to strictly follow protocols for investigating alarms from monitoring devices in ICUs. Regular blood tests are performed to help detect problems early, including daily sets of electrolytes and a CBC, as well as Mg, phosphate, Ca levels for patients with arrhythmias, and weekly liver enzymes and coagulation profiles for those receiving TPN. Point-of-care testing can be used to do certain blood tests at the patient's b

* Pass/Fail Grade: PASS

* Increasing tokens deliver minimal changes to answer quality or length; the model correctly answers the query but the answer itself is lengthy.

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
user_input_2 = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
generate_rag_response(user_input_2,temperature=0.1,top_p=0.8)

Llama.generate: prefix-match hit


"Answer:\nThe common symptoms of appendicitis include epigastric or periumbilical pain followed by brief nausea, vomiting, and anorexia. After a few hours, the pain shifts to the right lower quadrant. Pain increases with cough and motion. Classic signs are right lower quadrant direct and rebound tenderness located at McBurney's point, as well as pain felt in the right lower quadrant with palpation of the left lower quadrant (Rovsing sign), an increase in pain from passive extension of the right hip joint that stretches the iliopsoas muscle (psoas sign), or pain caused by passive internal rotation of the flexed thigh (obturator sign). Low-grade fever is common. However, these classic findings appear in less than 50% of patients, and many variations of symptoms and signs occur.\n\nAppendicitis cannot be cured via medicine alone. The standard treatment for appendicitis is surgical removal of the appendix through an open or laparoscopic appendectomy. This is because untreated appendicitis 

* Pass/Fail Grade: PASS

* Despite increasing temperature and decreasing top_p, the model delivers a similar answer.

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
user_input_3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
generate_rag_response(user_input_3,max_tokens=1024, temperature=0.1, top_p=0.5)

Llama.generate: prefix-match hit


"Answer:\nSudden patchy hair loss, also known as alopecia areata, is a common autoimmune disorder that affects genetically susceptible individuals. The exact cause of this condition is unclear but it's believed to involve environmental triggers and inflammation targeting the hair follicles.\n\nThe treatment options for alopecia areata include:\n1. Topical corticosteroids: These medications can be applied directly to the affected area to reduce inflammation and stimulate hair growth.\n2. Intralesional corticosteroid injections: Corticosteroids can also be injected into the affected area for more effective treatment.\n3. Systemic corticosteroids: In severe cases, oral corticosteroids may be prescribed to help reduce inflammation and promote hair growth.\n4. Topical minoxidil: This medication can be applied to the scalp to stimulate hair growth.\n5. Topical immunotherapy (diphencyprone or squaric acid dibutylester): These agents can help stimulate an immune response and promote hair regro

* Pass/Fail Grade: FAIL

* While the model answers both questions correctly, it answers them in the wrong order and possibly violates its system prompt.

## Output Evaluation

Let us now use the LLM-as-a-judge method to check the quality of the RAG system on two parameters - retrieval and generation. We illustrate this evaluation based on the answeres generated to the question from the previous section.

- We are using the same Mistral model for evaluation, so basically here the llm is rating itself on how well he has performed in the task.

In [ ]:
groundedness_rater_system_message = """
You are a highly analytical AI assistant evaluating other AI responses. Your task is to rate the 'Groundedness' of an AI-generated answer.
You will be provided with a Question, the Context used by the AI, and the AI's Answer.

Metric for Groundedness: The AI Answer must be *exclusively* derived from the provided Context. No external information should be used.

Evaluation Criteria (Score 1-5):
1 - Not grounded at all
2 - Grounded to a limited extent
3 - Grounded to a good extent
4 - Mostly grounded
5 - Fully grounded

Output Instructions:
1. State the evaluation steps.
2. Explain how the Answer's content aligns with (or deviates from) the Context to determine groundedness.
3. Provide the final score (1-5) and a brief justification.
"""

In [ ]:
relevance_rater_system_message = """
You are a highly analytical AI assistant evaluating other AI responses. Your task is to rate the 'Relevance' of an AI-generated answer.
You will be provided with a Question, the Context used by the AI, and the AI's Answer.

Metric for Relevance: The AI Answer must address all and only the important aspects of the Question, based *exclusively* on the provided Context.

Evaluation Criteria (Score 1-5):
1 - Not relevant at all
2 - Relevant to a limited extent
3 - Relevant to a good extent
4 - Mostly relevant
5 - Fully relevant

Output Instructions:
1. State the evaluation steps.
2. Explain how the Answer's content aligns with (or deviates from) the Question and Context to determine relevance.
3. Provide the final score (1-5) and a brief justification.
"""

In [ ]:
user_message_template = """
###Question
{question}

###Context
{context}

###Answer
{answer}
"""

In [ ]:
def generate_ground_relevance_response(user_input,k=3,max_tokens=512,temperature=0,top_p=0.95,top_k=50):
    global qna_system_message,qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks = retriever.get_relevant_documents(query=user_input,k=k)
    context_list = [d.page_content for d in relevant_document_chunks]
    context_for_query = ". ".join(context_list)

    # Combine user_prompt and system_message to create the prompt
    prompt = f"""[INST]{qna_system_message}\n
                {'user'}: {qna_user_message_template.format(context=context_for_query, question=user_input)}
                [/INST]"""

    response = llm(
            prompt=prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            echo=False
            )

    answer =  response["choices"][0]["text"]

    # Combine user_prompt and system_message to create the prompt
    groundedness_prompt = f"""[INST]{groundedness_rater_system_message}\n
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}
                [/INST]"""

    # Combine user_prompt and system_message to create the prompt
    relevance_prompt = f"""[INST]{relevance_rater_system_message}\n
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}
                [/INST]"""

    response_1 = llm(
            prompt=groundedness_prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            echo=False
            )

    response_2 = llm(
            prompt=relevance_prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            echo=False
            )

    return response_1['choices'][0]['text'],response_2['choices'][0]['text']

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
ground,rel = generate_ground_relevance_response(user_input="What is the protocol for managing sepsis in a critical care unit?", k=1, max_tokens=512)

print(ground,end="\n\n")

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 Evaluation steps:
1. Read the question and understand its meaning.
2. Read and comprehend the context provided.
3. Identify key information from the context that relates to the question.
4. Analyze the AI answer to determine if it is exclusively derived from the context.
5. Determine the groundedness score based on the alignment of the answer with the context.

Explanation:
The AI answer aligns closely with the context, as it mentions the importance of preventing infection and close monitoring in critical care units for managing sepsis. The answer also references the routine daily blood tests mentioned in the context for detecting problems early, which includes specific tests for sepsis such as electrolytes, CBC, magnesium, phosphate, calcium levels, and blood cultures. Furthermore, the AI answer mentions the use of complex devices for continuous monitoring and following strict protocols for investigating alarms, which is also mentioned in the context. The answer also acknowledges tha

In [ ]:
print(rel)

 Evaluation steps:
1. Read the question and understand its meaning.
2. Read the context and identify key concepts related to the question.
3. Analyze the AI answer to determine if it addresses all important aspects of the question based on the provided context.
4. Determine the relevance score (1-5) based on the analysis.

Explanation:
The question asks about the protocol for managing sepsis in a critical care unit, and the context provides information about critical care medicine, patient monitoring, blood tests, and infection control measures. The AI answer acknowledges the importance of preventing infections and close monitoring in managing sepsis, as well as the role of routine laboratory tests. It also mentions following ICU protocols for investigating alarms and specific population-based protocols. All these points align with the question and context, making the answer mostly relevant (score 4).

Justification:
The AI answer addresses all essential aspects of managing sepsis in a

* The model received a 5 for groundedness and a 4 for relevancy. This suggests the model does an excellent job retrieving information from the PDF and a solid job augmenting relevant information.

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
ground,rel = generate_ground_relevance_response(user_input="What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?", k=1, max_tokens=512)

print(ground,end="\n\n")

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 Evaluation steps:
1. Read the question and understand its requirements.
2. Analyze the context provided to determine if it contains sufficient information to generate a grounded answer.
3. Compare the content of the AI's answer with the context to assess the degree of alignment.
4. Determine the groundedness score based on the comparison.

Explanation:
The question asks about the common symptoms for appendicitis and whether it can be cured via medicine or if surgery is required. The context provided in the text discusses the etiology, symptoms, signs, diagnosis, and treatment options for appendicitis.

The AI's answer accurately summarizes the common symptoms of appendicitis from the context and explains that appendicitis cannot be cured via medicine alone and requires surgical intervention. The answer also mentions the standard procedure for treating appendicitis, which is an appendectomy.

Therefore, the AI's answer aligns closely with the information provided in the context, and no

In [ ]:
print(rel)

 Evaluation steps:
1. Read the question carefully to understand its main components and requirements.
2. Analyze the context provided to identify relevant information related to the question.
3. Compare the question and context to determine how well the AI answer addresses the important aspects of the question based on the context.
4. Evaluate the relevance of the AI answer using the given criteria (1-5).

Explanation:
The question asks about common symptoms for appendicitis and whether it can be cured via medicine or if surgery is required. The context provided includes information on the etiology, symptoms, signs, diagnosis, and treatment options for appendicitis.

The AI answer addresses all the important aspects of the question by listing common symptoms for appendicitis and stating that it cannot be cured with medicine alone. Additionally, the answer explains that surgery (appendectomy) is necessary if left untreated, which aligns with the context's information on potential compli

* The model received a 5 for groundedness and a 5 for relevancy. This suggests the model does an excellent job retrieving information from the PDF and an excellent job augumenting relevant information.

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
ground,rel = generate_ground_relevance_response(user_input="What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?", k=1, max_tokens=512)

print(ground,end="\n\n")

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 Evaluation steps:
1. Read the question carefully to understand what is being asked.
2. Read the context provided to identify any relevant information that could be used to answer the question.
3. Compare the content of the AI's answer to the context to determine if the answer is exclusively derived from the context.

Explanation:
The AI's answer acknowledges the limitations of its ability to provide medical advice or suggest specific treatments based on the user's description of sudden patchy hair loss. However, it correctly identifies alopecia areata as a condition characterized by sudden, patchy hair loss and lists treatment options for this condition from the context provided. The answer aligns closely with the context and is grounded to a good extent (score 3).

Justification:
The AI's answer accurately reflects the information in the context regarding alopecia areata and its treatment options. While it does not provide specific medical advice or suggest treatments for the user, i

In [ ]:
print(rel)

 Evaluation steps:
1. Understand the Question and Context.
2. Identify the key points in the Question and Context related to the topic of sudden patchy hair loss and its possible causes and treatments.
3. Analyze how well the AI Answer addresses these key points based on the provided Context.
4. Determine the relevance score (1-5) for the AI Answer.

Explanation:
The Question asks about effective treatments or solutions for sudden patchy hair loss and possible causes behind it. The Context provides information on various types of hair loss, including alopecia areata, which is characterized by sudden, patchy hair loss. The Context also mentions treatment options for alopecia areata, such as topical corticosteroids, oral antimalarials, corticosteroids, retinoids, or immunosuppressants.

The AI Answer acknowledges the limitations of providing medical advice or suggesting specific treatments based on the description of sudden patchy hair loss but correctly identifies alopecia areata as a c

* The LLM-As-Judge takes a harsher approach here, grading a 3 for groundedness and a 4 for relativity.

* The prompts for groundedness and relativity are worth revisiting; the system is using inconsistent criteria while providing grades.

### Query 4: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
ground,rel = generate_ground_relevance_response(user_input="What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?", k=1, max_tokens=512)

print(ground,end="\n\n")

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 Evaluation steps:
1. Read and understand the question, context, and AI answer.
2. Identify how the AI answer aligns or deviates from the context in terms of content related to the question.
3. Determine the groundedness score based on the extent of alignment between the AI answer and the context.

Explanation:
The AI answer acknowledges the importance of early intervention by rehabilitation specialists for individuals with brain injuries, including TBI, as stated in the context. The answer also mentions that rehabilitation therapy varies depending on the severity and location of the injury, which is consistent with the context. Furthermore, the answer correctly identifies some potential secondary disabilities that may need prevention, such as pressure ulcers and joint contractures.

The AI answer does not suggest any specific treatments or therapies beyond what is mentioned in the context. It also provides a disclaimer stating that it cannot provide medical advice or suggest specific 

In [ ]:
print(rel)

 Evaluation steps:
1. Understand the Question and Context.
2. Identify the key points in the Question and Context related to the treatment of brain injuries.
3. Analyze how well the AI Answer addresses those key points based on the Context.
4. Determine the relevance score (1-5) based on the analysis.

Explanation:
The Question asks about treatments for a person with a brain injury resulting in temporary or permanent impairment of brain function. The Context discusses the importance of early intervention by rehabilitation specialists, prevention of secondary disabilities, and cognitive therapy for those with severe cognitive dysfunction.

The AI Answer acknowledges that it cannot provide medical advice or suggest specific treatments but shares information about the importance of early intervention by rehabilitation specialists and the various types of therapy they may provide based on the severity and location of the injury. The Answer also mentions the need to prevent secondary disabi

* The model provides a 5 for groundedness and a 4 for relativity.

* Opportunity to enhance LLM-as-Judge prompts by defining a consistent format for outputs.

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
ground,rel = generate_ground_relevance_response(user_input="What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?", k=1, max_tokens=512)

print(ground,end="\n\n")

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 Evaluation steps:
1. Read the question to understand what is being asked.
2. Read the context to identify any relevant information related to the question.
3. Analyze the AI's answer to determine if it aligns with or deviates from the context.
4. Determine the groundedness of the AI's answer based on the alignment with the context.

Explanation:
The AI's answer outlines several precautions and treatment steps for a person who has fractured their leg during a hiking trip, which aligns well with the context provided. The context mentions immobilization, pain management, definitive treatment (reduction), RICE method, and seeking medical attention promptly. The AI's answer covers all these aspects in detail.

Therefore, based on the alignment with the context, I would rate the groundedness of the AI's answer as 5 (Fully grounded). The AI's response is entirely derived from the provided context and does not include any external information.



In [ ]:
print(rel)

 Evaluation steps:
1. Understand the question and context.
2. Identify the key points in the question and context related to the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip.
3. Determine how well the AI's answer addresses these key points based on the provided context.
4. Evaluate the relevance of the AI's answer to the question.

Explanation:
The AI's answer outlines several necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, which aligns well with the question. The context provided discusses various treatments for injuries, including splinting, reduction, RICE method, and seeking medical attention. The AI's answer covers all these aspects, making it mostly relevant to the question.

Final score: 4 - Mostly relevant
Justification: The AI's answer addresses all the essential points from the question and context, but it could have been more specific about some details, such a

* The model provides a 5 for groundedness and a 4 for relativity.

## Actionable Insights and Business Recommendations

* Vector database creation time increases with the number of pages in the PDF document.

* Max_tokens depends on query complexity; in complex prompts, the system filled the available token space regardless of number of tokens.

* Refine prompt design to shorten response length and refine temperature to increase creativity, encourgaging the model to summarize with shorter answers.

* Continuously adjust RAG parameters based on specific use cases for optimal performance.

* Continue prioritizing groundedness and relativity as output metrics; evolve the LLM-As-Judge prompts to ensure the model follows a consistent format for each grade.


<font size=6 color='blue'>Power Ahead</font>
___